# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnanyaGujral/flyrank-ml-internship-start/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

A ranked queue with human-readable reason codes, plus the limits of the tool. Decision-support, not a forecast.

## 1. Ranked actions + reason codes

**What this queue is.** Every already-visible page in the pool, ranked, with a four-way action and a
feature-only reason code. The ranking quality is *measured* only on held-out clients (see `w05_model.ipynb`);
everything else about the queue is *decision-support*.

**Reason codes** (never touch the label): `ctr_gap` (CTR ≤ half the pool median), `high_exposure`
(prior impressions in the pool's top quartile), `no_clicks` (zero prior clicks at real exposure),
`base_anchor` (none of the above).

**The four actions** (by score percentile): `investigate_first` (top), `watch_prepare`, `monitor`,
`no_action` (rest).

In [1]:
# ---- shared offline setup: bundled slice, leak-free pool + features, transparent rule ----
import json
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
MIN_PREV_IMPRESSIONS = 100     # "already visible" floor on the TRAILING window
DROP_FACTOR = 0.8              # declined = next-30 impressions < 80% of prior-30

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
DATA = root / "data" / "raw" / "content_refresh_anonymized.csv"
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(DATA)


def build_pool(frame):
    """Already-visible pages with a usable trailing window, plus leak-free features.

    Every feature is measured in the PRIOR 30-day window (days 31-60) or is static
    metadata. Nothing is taken from the 90-day totals or the outcome window, because
    those overlap the label's comparison window.
    """
    pool = frame[frame["impressions_prev_30d"] >= MIN_PREV_IMPRESSIONS].copy()
    pool["declined_next_30d"] = (
        pool["impressions_last_30d"] < DROP_FACTOR * pool["impressions_prev_30d"]).astype(int)
    pool["log_impr_prev"] = np.log1p(pool["impressions_prev_30d"])
    pool["log_clk_prev"] = np.log1p(pool["clicks_prev_30d"])
    pool["log_sess_prev"] = np.log1p(pool["sessions_prev_30d"])
    pool["ctr_prev"] = 100 * pool["clicks_prev_30d"] / pool["impressions_prev_30d"]
    pool["clk_per_sess_prev"] = pool["clicks_prev_30d"] / pool["sessions_prev_30d"].replace(0, np.nan)
    pool["has_keyword"] = pool["search_volume"].notna().astype(int)
    pool["has_clicks_prev"] = (pool["clicks_prev_30d"] > 0).astype(int)
    pool["has_sessions_prev"] = (pool["sessions_prev_30d"] > 0).astype(int)
    g = pool.groupby("client_id")
    pool["ctr_prev_z"] = ((pool["ctr_prev"] - g["ctr_prev"].transform("mean"))
                          / g["ctr_prev"].transform("std").replace(0, np.nan))
    pool["impr_prev_share"] = pool["impressions_prev_30d"] / g["impressions_prev_30d"].transform("sum")
    pool["clk_prev_share"] = pool["clicks_prev_30d"] / g["clicks_prev_30d"].transform("sum").replace(0, np.nan)
    return pool


NUM_FEATURES = ["log_impr_prev", "log_clk_prev", "log_sess_prev", "ctr_prev", "clk_per_sess_prev",
                "ctr_prev_z", "impr_prev_share", "clk_prev_share",
                "search_volume", "competition", "cpc", "word_count", "char_count",
                "content_age_days", "days_since_last_update",
                "has_keyword", "has_clicks_prev", "has_sessions_prev"]
CAT_FEATURES = ["competition_level", "content_type", "main_intent", "age_tier",
                "freshness_tier", "word_count_tier"]


def feature_matrix(pool):
    Xn = pool[NUM_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([Xn, pd.get_dummies(pool[CAT_FEATURES].astype(str), dtype=float)], axis=1)


def rulescore(pool):
    """Transparent rule: low trailing CTR, weighted by trailing exposure."""
    med = float(pool["ctr_prev"].median())
    return (pool["ctr_prev"] <= 0.5 * med).to_numpy() * np.log1p(pool["impressions_prev_30d"].to_numpy())


pool = build_pool(raw)
X = feature_matrix(pool)
y = pool["declined_next_30d"].to_numpy()
groups = pool["client_id"].to_numpy()
rule = rulescore(pool)
print(f"bundled slice: {len(raw):,} rows -> pool {len(pool):,} pages | "
      f"{pool['client_id'].nunique()} pseudo-clients")
print(f"decline base rate in pool: {y.mean():.3f} | feature matrix: {X.shape}")
print("label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)")


def pk(table_scores, y_true, ks=(10, 20, 50, 100)):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(table_scores))
    return {str(k): round(float(y_true[order[:k]].mean()), 3) for k in ks}

bundled slice: 30,000 rows -> pool 18,010 pages | 30 pseudo-clients
decline base rate in pool: 0.616 | feature matrix: (18010, 43)
label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)


In [2]:
def pk(table_scores, y_true, ks=(10, 20, 50, 100)):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(table_scores))
    return {str(k): round(float(y_true[order[:k]].mean()), 3) for k in ks}

In [3]:
median_ctr = float(pool["ctr_prev"].median())
q75 = float(pool["impressions_prev_30d"].quantile(0.75))

def codes(i):
    out = []
    if pool["ctr_prev"].iloc[i] <= 0.5 * median_ctr:
        out.append("ctr_gap")
    if pool["clicks_prev_30d"].iloc[i] == 0 and pool["impressions_prev_30d"].iloc[i] >= 500:
        out.append("no_clicks")
    if pool["impressions_prev_30d"].iloc[i] >= q75:
        out.append("high_exposure")
    return "+".join(out) if out else "base_anchor"

play = pool.copy()
play["score"] = rule
play["reason_code"] = [codes(i) for i in range(len(play))]
pct = play["score"].rank(pct=True, ascending=False)
play["action"] = np.select(
    [pct <= 0.10, pct <= 0.25, pct <= 0.50],
    ["investigate_first", "watch_prepare", "monitor"], default="no_action")
mix = play["action"].value_counts().to_dict()
print("action mix:", {k: int(mix.get(k, 0)) for k in ["investigate_first", "watch_prepare", "monitor", "no_action"]})
print("top reason codes in investigate_first:")
print(play[play["action"] == "investigate_first"]["reason_code"].value_counts().head(5).to_string())

action mix: {'investigate_first': 1801, 'watch_prepare': 2709, 'monitor': 3127, 'no_action': 10373}
top reason codes in investigate_first:
reason_code
ctr_gap+no_clicks                  881
ctr_gap+high_exposure              577
ctr_gap+no_clicks+high_exposure    191
ctr_gap                            152


## 2. Intended use and limits

**Intended use.** An editor opens the top of the queue, reads the reason code, and forms a hypothesis
about one page at a time. The queue orders; the human decides.

**Where it stops being valid.**

In [4]:
review = play[play["action"] == "investigate_first"]
print(f"review-first tier: {len(review):,} pages, covering "
      f"{100*review['impressions_prev_30d'].sum()/play['impressions_prev_30d'].sum():.1f}% of pool impressions")
print(f"measured ranking quality is only the held-out-client P@K (w05/w06) - the rest is ordering, not proof")
limits = [
    "one snapshot: no cross-month replication",
    "the label is a window comparison, so seasonal pages can look like decliners",
    "scores are rankings, not calibrated probabilities",
    "the queue must not be reused after the next window closes",
]
for lim in limits:
    print(" -", lim)

review-first tier: 1,801 pages, covering 15.7% of pool impressions
measured ranking quality is only the held-out-client P@K (w05/w06) - the rest is ordering, not proof
 - one snapshot: no cross-month replication
 - the label is a window comparison, so seasonal pages can look like decliners
 - scores are rankings, not calibrated probabilities
 - the queue must not be reused after the next window closes


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [5]:
checks = [
    "read the page: is the low CTR a real problem or a deliberate intent?",
    "check for a campaign / launch / seasonality window that explains the movement",
    "confirm the metric change is not instrumentation",
    "confirm the page is not already mid-edit",
]
nogo = ["auto-rewriting content", "auto-deleting or pruning", "acting on a single score with no review",
        "using the queue as a traffic forecast"]
print("human review checklist:")
for c in checks:
    print("  [ ]", c)
print("\nno-go list (never automate):")
for n in nogo:
    print("  x", n)
print(f"\nsample of the review-first tier (aggregates only, no identifiers):")
print(review[["impressions_prev_30d", "clicks_prev_30d", "ctr_prev", "days_since_last_update"]]
      .describe().loc[["mean", "50%", "max"]].round(2).to_string())

human review checklist:
  [ ] read the page: is the low CTR a real problem or a deliberate intent?
  [ ] check for a campaign / launch / seasonality window that explains the movement
  [ ] confirm the metric change is not instrumentation
  [ ] confirm the page is not already mid-edit

no-go list (never automate):
  x auto-rewriting content
  x auto-deleting or pruning
  x acting on a single score with no review
  x using the queue as a traffic forecast

sample of the review-first tier (aggregates only, no identifiers):
      impressions_prev_30d  clicks_prev_30d  ctr_prev  days_since_last_update
mean               4638.79             1.14      0.01                   59.07
50%                2119.00             0.00      0.00                   25.00
max              111885.00            61.00      0.06                  194.00


## 4. Monitoring / retrain triggers

*What would tell me the recommendations went stale?*

In [6]:
band = (round(float(y.mean()) - 0.05, 3), round(float(y.mean()) + 0.05, 3))
print(f"base-rate trigger band: pause and re-check if the pool decline base rate leaves {band}")
print(f"held-out P@50 floor: re-examine if a re-run drops below {round(float(pk(rule, y)['50']) - 0.10, 2)}")
print("ownership: a fixed seed-42 grouped split, re-run on the next window before any release")

base-rate trigger band: pause and re-check if the pool decline base rate leaves (0.566, 0.666)
held-out P@50 floor: re-examine if a re-run drops below 0.7
ownership: a fixed seed-42 grouped split, re-run on the next window before any release


## 5. Exports for the paper

*Write the queue and the receipt to `work/outputs/` — the paper builds on these files.*

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig_dir = OUT / "figures"
fig_dir.mkdir(exist_ok=True)
order = ["investigate_first", "watch_prepare", "monitor", "no_action"]
vals = [100 * mix.get(a, 0) / len(play) for a in order]
fig, ax = plt.subplots(figsize=(6.5, 3.6))
bars = ax.bar(order, vals, color=["#d93025", "#f9ab00", "#4285F4", "#9aa0a6"])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("% of pool"); ax.set_title("The four actions and their share of the pool")
ax.grid(alpha=0.3, axis="y"); fig.tight_layout()
fig.savefig(fig_dir / "action_mix.png", dpi=110); plt.close(fig)

queue_path = OUT / "action_playbook_queue.csv"
play.sort_values("score", ascending=False)[["content_id", "client_id", "score", "action", "reason_code",
                                           "impressions_prev_30d", "ctr_prev"]].to_csv(queue_path, index=False)
print("figure written:", (fig_dir / "action_mix.png").name)
print(f"queue written: {queue_path.name} | rows {len(play):,} (gitignored)")

metrics = {
    "task": "ml-10",
    "data": "data/raw/content_refresh_anonymized.csv",
    "decision_support_queue": {"rows": int(len(play)), "action_mix": {k: int(mix.get(k, 0)) for k in order},
                              "review_first_share_of_demand": round(float(review['impressions_prev_30d'].sum()
                                                                         / play['impressions_prev_30d'].sum()), 3)},
    "top_reason_codes_review_first": play[play["action"] == "investigate_first"]["reason_code"].value_counts().head(6).to_dict(),
    "measured_claims": {"heldout_p50": None, "note": "see w05/w06 receipts for the measured ranking quality"},
    "limits": ["one snapshot", "window-comparison label", "scores are rankings not probabilities"],
    "monitoring": {"base_rate_trigger_band": list(band)},
}
(OUT / "w07_playbook_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("receipt written: work/outputs/w07_playbook_metrics.json")

figure written: action_mix.png
queue written: action_playbook_queue.csv | rows 18,010 (gitignored)
receipt written: work/outputs/w07_playbook_metrics.json


### Self-check

- [x] Ranked actions with human-readable, feature-only reason codes
- [x] Intended use and the exact places it stops being valid
- [x] Human-review checklist and an explicit no-go list
- [x] Monitoring / retrain triggers written as numbers
- [x] Queue + figure exported to `work/outputs/` (gitignored), receipt committed
- [x] Committed under `work/notebooks/`